[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/28_neural_networks/02_initialization_and_normalization.ipynb)

# Bài 02 — Khởi tạo trọng số và chuẩn hoá

**🎯 Mục tiêu**
- Vì sao khởi tạo quan trọng: **đối xứng** khi khởi tạo bằng 0; tín hiệu **tắt dần / bùng nổ** qua nhiều lớp.
- **Xavier/Glorot** và **He/Kaiming**: dẫn xuất từ việc giữ phương sai ổn định.
- Chuẩn hoá đầu vào; **Batch Normalization** from scratch (train vs eval), **Layer Normalization**.
- Thí nghiệm: mạng sâu có/không BN; các lỗi thường gặp.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/28_neural_networks

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
rng = np.random.default_rng(0)

## 1. Khởi tạo bằng 0: đối xứng không bao giờ bị phá

Nếu mọi trọng số bằng nhau, mọi nơ-ron trong một lớp tính **cùng một** giá trị và nhận **cùng một** gradient → sau mọi bước cập nhật chúng vẫn giống hệt nhau: lớp 100 nơ-ron chỉ hoạt động như 1 nơ-ron.

In [ ]:
X_dg, y_dg = load_digits(return_X_y=True)
X_t = torch.tensor(StandardScaler().fit_transform(X_dg), dtype=torch.float32)
y_t = torch.tensor(y_dg)

model0 = nn.Sequential(nn.Linear(64, 32), nn.Tanh(), nn.Linear(32, 10))
for p in model0.parameters():
    nn.init.constant_(p, 0.1)                                           # mọi trọng số bằng nhau
opt = torch.optim.SGD(model0.parameters(), lr=0.1)
for _ in range(100):
    opt.zero_grad()
    nn.functional.cross_entropy(model0(X_t), y_t).backward()
    opt.step()
W = model0[0].weight.detach()
print(f"sau 100 bước: độ lệch chuẩn giữa 32 nơ-ron ẩn (theo từng trọng số) = {W.std(dim=0).max().item():.2e} → các nơ-ron vẫn giống hệt nhau")
print(f"accuracy train = {(model0(X_t).argmax(1) == y_t).float().mean().item():.3f}")

→ Khởi tạo phải **ngẫu nhiên** để phá đối xứng. Nhưng ngẫu nhiên với độ lớn bao nhiêu?

## 2. Tín hiệu qua nhiều lớp: tắt dần hay bùng nổ

Một nơ-ron $z = \sum_{i=1}^{n}w_i x_i$ với $w_i, x_i$ độc lập, trung bình 0: $\text{Var}(z) = n\,\text{Var}(w)\,\text{Var}(x)$. Qua mỗi lớp, phương sai nhân với $n\,\text{Var}(w)$:
- $n\,\text{Var}(w) < 1$ → tín hiệu **tắt dần** về 0 (gradient cũng vậy);
- $n\,\text{Var}(w) > 1$ → **bùng nổ** (tanh bão hoà ở ±1).

Truyền một batch ngẫu nhiên qua 20 lớp, mỗi lớp 512 nơ-ron, đo độ lệch chuẩn của đầu ra từng lớp:

In [ ]:
def activation_stds(act, weight_std_fn, depth=20, width=512, seed=0):
    r = np.random.default_rng(seed)
    h = r.normal(size=(1000, width))
    stds = []
    for _ in range(depth):
        W = r.normal(0, weight_std_fn(width), (width, width))
        h = act(h @ W)
        stds.append(h.std())
    return stds


relu = lambda z: np.maximum(0, z)
setups = {
    "tanh, std = 0.01": (np.tanh, lambda n: 0.01),
    "tanh, std = 1": (np.tanh, lambda n: 1.0),
    "tanh, Xavier √(1/n)": (np.tanh, lambda n: np.sqrt(1 / n)),
    "ReLU, Xavier √(1/n)": (relu, lambda n: np.sqrt(1 / n)),
    "ReLU, He √(2/n)": (relu, lambda n: np.sqrt(2 / n)),
}
fig, ax = plt.subplots(figsize=(10, 4.5))
for name, (act, fn) in setups.items():
    stds = activation_stds(act, fn)
    ax.semilogy(range(1, 21), np.maximum(stds, 1e-30), marker=".", label=name)
    print(f"{name:22s} std lớp 1 = {stds[0]:.3e} | lớp 20 = {stds[-1]:.3e}")
ax.set_xlabel("lớp")
ax.set_ylabel("std của kích hoạt (log)")
ax.legend(fontsize=8)
plt.show()

- std 0.01: tín hiệu tắt về 0 sau vài lớp → gradient cũng tắt.
- std 1 với tanh: kích hoạt dồn về ±1 (bão hoà) → gradient ≈ 0.
- **Xavier** ($\text{Var}(w) = 1/n$) giữ tanh ở cùng bậc độ lớn (chỉ giảm chậm — vì tanh nén giá trị, PyTorch khuyến nghị nhân thêm hệ số `gain = 5/3`); nhưng với ReLU (cắt nửa âm, chỉ giữ một nửa phương sai) tín hiệu giảm theo hàm mũ.
- **He** ($\text{Var}(w) = 2/n$) bù hệ số 2 đó → ReLU ổn định qua 20 lớp.

| Khởi tạo | $\text{Var}(w)$ | Dùng với |
|---|---|---|
| Xavier/Glorot (2010) | $\frac{1}{n_{in}}$ hoặc $\frac{2}{n_{in} + n_{out}}$ | tanh, sigmoid, tuyến tính |
| He/Kaiming (2015) | $\frac{2}{n_{in}}$ | ReLU và họ ReLU |
| PyTorch `nn.Linear` mặc định | uniform, $\text{Var} = \frac{1}{3n_{in}}$ | "an toàn" nhưng hơi nhỏ cho ReLU sâu |

`nn.init.xavier_normal_`, `nn.init.kaiming_normal_(w, nonlinearity="relu")` trong PyTorch; bias thường khởi tạo 0.

## 3. Chuẩn hoá đầu vào

Cùng lý do: nếu feature có thang đo rất khác nhau, mặt loss **kéo dài** theo một số hướng → một learning rate vừa với feature lớn thì quá nhỏ với feature nhỏ và ngược lại; gradient descent zíc-zắc. Dữ liệu khách hàng rời bỏ có độ lệch chuẩn từ ~1 (số sản phẩm) tới ~1 400 (tổng chi tiêu):

In [ ]:
churn = pd.read_csv("../../data/sample/customer_churn.csv")
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
print("độ lệch chuẩn từng feature:", churn[num_cols].std().round(1).to_dict())
Xc_raw = torch.tensor(churn[num_cols].to_numpy(float), dtype=torch.float32)
Xc_std = torch.tensor(StandardScaler().fit_transform(churn[num_cols]), dtype=torch.float32)
yc = torch.tensor(churn["churn"].to_numpy(), dtype=torch.float32)


def churn_loss_after(X, lr, epochs=20, seed=0):
    torch.manual_seed(seed)
    m = nn.Sequential(nn.Linear(5, 32), nn.ReLU(), nn.Linear(32, 1))
    o = torch.optim.SGD(m.parameters(), lr=lr)
    g = torch.Generator().manual_seed(seed)
    for _ in range(epochs):
        perm = torch.randperm(len(X), generator=g)
        for s in range(0, len(perm), 64):
            idx = perm[s:s + 64]
            o.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(X[idx]).squeeze(1), yc[idx]).backward()
            o.step()
    with torch.no_grad():
        return nn.functional.binary_cross_entropy_with_logits(m(X).squeeze(1), yc).item()


for lr in [1e-4, 1e-3, 1e-2, 1e-1]:
    print(f"lr = {lr:<6}: loss sau 20 epoch — feature thô = {churn_loss_after(Xc_raw, lr):.3f} | đã chuẩn hoá = {churn_loss_after(Xc_std, lr):.3f}")

Không learning rate nào giúp bản dùng feature thô đạt tới loss của bản đã chuẩn hoá: với feature thô, learning rate đủ nhỏ để không "nổ" theo `total_charges` thì quá nhỏ để học từ `n_products`. Chuẩn hoá (trung bình 0, độ lệch chuẩn 1 — fit trên **train**) gần như luôn cần cho mạng nơ-ron. (Với ảnh, mọi pixel cùng thang đo — chia 255 hoặc chuẩn hoá theo kênh là đủ.)

## 4. Batch Normalization

Ngay cả khi đầu vào đã chuẩn hoá, phân phối kích hoạt của các lớp **bên trong** thay đổi liên tục trong quá trình huấn luyện. **Batch Norm** (Ioffe & Szegedy, 2015) chuẩn hoá đầu ra của mỗi lớp theo **mini-batch**, rồi cho mạng tự học lại thang đo:
$$\hat z = \frac{z - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \qquad y = \gamma\,\hat z + \beta$$
- Khi **train**: $\mu_B, \sigma_B^2$ tính trên batch hiện tại; đồng thời cập nhật **trung bình trượt** (running mean/var).
- Khi **eval/inference**: dùng running mean/var (dự đoán một mẫu không phụ thuộc mẫu khác trong batch).
- $\gamma, \beta$ học được — BN có thể tự "tắt" chính nó nếu cần.

In [ ]:
class BatchNorm1dScratch:
    def __init__(self, n_features, momentum=0.1, eps=1e-5):
        self.gamma, self.beta = np.ones(n_features), np.zeros(n_features)
        self.running_mean, self.running_var = np.zeros(n_features), np.ones(n_features)
        self.momentum, self.eps, self.training = momentum, eps, True

    def forward(self, z):
        if self.training:
            mu, var = z.mean(axis=0), z.var(axis=0)                     # var chia n (biased) để chuẩn hoá
            n = z.shape[0]
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mu
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var * n / (n - 1)  # unbiased cho running
        else:
            mu, var = self.running_mean, self.running_var
        return self.gamma * (z - mu) / np.sqrt(var + self.eps) + self.beta


z_batch = rng.normal(5, 3, (64, 4))
mine = BatchNorm1dScratch(4)
ref = nn.BatchNorm1d(4)
out_mine = mine.forward(z_batch)
out_ref = ref(torch.tensor(z_batch, dtype=torch.float32)).detach().numpy()
print(f"train mode — sai lệch lớn nhất so với nn.BatchNorm1d: {np.abs(out_mine - out_ref).max():.2e}")
print(f"running_mean sau 1 batch: tự viết {mine.running_mean.round(3)} | PyTorch {ref.running_mean.numpy().round(3)}")
mine.training = False
ref.eval()
z_new = rng.normal(5, 3, (3, 4))
print(f"eval mode — sai lệch: {np.abs(mine.forward(z_new) - ref(torch.tensor(z_new, dtype=torch.float32)).detach().numpy()).max():.2e}")

### Train vs eval: một lỗi kinh điển

Ở chế độ **train**, đầu ra của một mẫu phụ thuộc các mẫu khác trong batch. Quên `model.eval()` khi dự đoán → kết quả thay đổi theo batch:

In [ ]:
torch.manual_seed(0)
bn_model = nn.Sequential(nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Linear(32, 10))
for _ in range(50):                                                     # làm nóng running stats
    bn_model(X_t[torch.randint(0, len(X_t), (64,))])
x0 = X_t[:1]
with torch.no_grad():
    bn_model.train()
    out_a = bn_model(torch.cat([x0, X_t[1:32]]))[0]
    out_b = bn_model(torch.cat([x0, X_t[500:531]]))[0]
    bn_model.eval()
    out_c = bn_model(torch.cat([x0, X_t[1:32]]))[0]
    out_d = bn_model(torch.cat([x0, X_t[500:531]]))[0]
print(f"train mode: cùng ảnh, khác batch → chênh lệch logit = {(out_a - out_b).abs().max().item():.3f}")
print(f"eval mode : cùng ảnh, khác batch → chênh lệch logit = {(out_c - out_d).abs().max().item():.3f}")

## 5. Thí nghiệm: mạng sâu có và không có Batch Norm

MLP 10 lớp ẩn × 64 (ReLU), SGD + momentum (lr = 0.01, 20 epoch). Ba cấu hình: khởi tạo mặc định của PyTorch; khởi tạo He; khởi tạo mặc định + Batch Norm:

In [ ]:
X_tr, X_va, y_tr, y_va = train_test_split(X_t, y_t, test_size=0.25, random_state=0, stratify=y_t)


def deep_net(use_bn, he_init, depth=10, width=64, seed=0):
    torch.manual_seed(seed)
    layers, d = [], 64
    for _ in range(depth):
        lin = nn.Linear(d, width)
        if he_init:
            nn.init.kaiming_normal_(lin.weight, nonlinearity="relu")
            nn.init.zeros_(lin.bias)
        layers.append(lin)
        if use_bn:
            layers.append(nn.BatchNorm1d(width))
        layers.append(nn.ReLU())
        d = width
    return nn.Sequential(*layers, nn.Linear(d, 10))


def run(use_bn, he_init, lr=0.01, epochs=20, seed=0):
    model = deep_net(use_bn, he_init, seed=seed)
    o = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    g = torch.Generator().manual_seed(seed)
    accs = []
    for _ in range(epochs):
        model.train()
        perm = torch.randperm(len(X_tr), generator=g)
        for s in range(0, len(perm), 32):
            idx = perm[s:s + 32]
            o.zero_grad()
            nn.functional.cross_entropy(model(X_tr[idx]), y_tr[idx]).backward()
            o.step()
        model.eval()
        with torch.no_grad():
            accs.append((model(X_va).argmax(1) == y_va).float().mean().item())
    return accs


fig, ax = plt.subplots(figsize=(10, 4.5))
for name, use_bn, he_init in [("khởi tạo mặc định", False, False), ("khởi tạo He", False, True), ("mặc định + Batch Norm", True, False)]:
    accs = run(use_bn, he_init)
    ax.plot(range(1, 21), accs, marker=".", label=name)
    print(f"{name:22s}: val accuracy sau epoch 1 / 5 / 20 = {accs[0]:.3f} / {accs[4]:.3f} / {accs[-1]:.3f}")
ax.set_xlabel("epoch")
ax.set_ylabel("val accuracy")
ax.legend()
plt.show()

Với khởi tạo mặc định, tín hiệu tắt dần qua 10 lớp và mạng **không học được gì** (≈ 10% — đoán mò). Hai cách sửa: khởi tạo đúng (He) giữ tín hiệu ổn định; hoặc Batch Norm chuẩn hoá lại kích hoạt ở mỗi lớp — mạng học được dù khởi tạo kém, và học nhanh nhất ở những epoch đầu. BN còn giúp chịu được learning rate lớn hơn; cơ chế chính xác vẫn còn tranh luận ("internal covariate shift" ban đầu; các nghiên cứu sau cho thấy BN làm mặt loss **trơn** hơn — Santurkar et al., 2018), nhưng hiệu quả thực nghiệm thì rõ.

## 6. Layer Normalization

**LayerNorm** (Ba et al., 2016) chuẩn hoá theo **các feature của từng mẫu** (không theo batch):
- không phụ thuộc kích thước batch, train/eval giống nhau;
- dùng trong RNN và **Transformer** (chương 31), nơi batch nhỏ hoặc độ dài chuỗi thay đổi.

In [ ]:
z = torch.tensor(rng.normal(5, 3, (4, 6)), dtype=torch.float32)
bn_out = nn.BatchNorm1d(6)(z)
ln_out = nn.LayerNorm(6)(z)
print("BatchNorm: trung bình theo CỘT (feature) ≈ 0:", bn_out.mean(dim=0).detach().numpy().round(4))
print("LayerNorm: trung bình theo HÀNG (mẫu) ≈ 0:  ", ln_out.mean(dim=1).detach().numpy().round(4))

| | BatchNorm | LayerNorm |
|---|---|---|
| Chuẩn hoá theo | batch (mỗi feature) | feature (mỗi mẫu) |
| Phụ thuộc batch | có — batch nhỏ thì nhiễu | không |
| Train ≠ eval | có (running stats) | không |
| Dùng nhiều trong | CNN (chương 29) | RNN, Transformer |

**Lưu ý thực hành:** BN cần batch đủ lớn (≥ 16–32); đặt trước hoặc sau kích hoạt đều gặp (bài gốc đặt trước); khi dùng BN, bias của lớp Linear/Conv ngay trước là thừa (β đảm nhận); luôn `model.eval()` khi đánh giá.

## 🧠 Kiểm tra nhanh

1. Vì sao không được khởi tạo mọi trọng số bằng 0 (hoặc bằng nhau)?
2. Dẫn xuất ngắn gọn vì sao Xavier chọn $\text{Var}(w) = 1/n$, He chọn $2/n$.
3. Batch Norm làm gì khi train và khi eval? Vì sao cần running mean/var?
4. Quên `model.eval()` với mạng có BN gây ra hiện tượng gì?
5. Khi nào dùng LayerNorm thay cho BatchNorm?

## 📌 Tóm tắt

- Khởi tạo ngẫu nhiên phá đối xứng; giữ phương sai ổn định qua lớp: Xavier (tanh/sigmoid), He (ReLU).
- Chuẩn hoá đầu vào; BN chuẩn hoá kích hoạt theo batch (train) / running stats (eval), giúp mạng sâu học nhanh, chịu LR lớn.
- LayerNorm chuẩn hoá theo mẫu — chuẩn của Transformer.

## 📚 Tài liệu tham khảo

- Glorot & Bengio (2010) — *Understanding the difficulty of training deep feedforward neural networks*
- He, Zhang, Ren & Sun (2015) — *Delving Deep into Rectifiers*
- Ioffe & Szegedy (2015) — *Batch Normalization*
- Ba, Kiros & Hinton (2016) — *Layer Normalization*
- Santurkar et al. (2018) — *How Does Batch Normalization Help Optimization?*